# Tarea Práctica: Calidad de Datos en IoT (Smart Building)

## Contexto
Estás trabajando como Data Engineer para una empresa de edificios inteligentes. 
Tenéis una red de sensores de temperatura y humedad desplegada en las oficinas. 
Sin embargo, la red es inestable: a veces los sensores pierden conexión, envían datos duplicados o se descalibran y mandan valores imposibles.

## Objetivo
Tu misión es limpiar el dataset raw (sucio) para que pueda ser usado por el equipo de Data Science para predecir el consumo energético.

Debes resolver los siguientes problemas:
1.  **Completitud:** Gestionar los datos faltantes.
2.  **Consistencia:** Eliminar duplicados.
3.  **Precisión:** Filtrar valores erróneos (Outliers).

### !IMPORATANTE, SI TOMAS UNA DECISIÓN , QUIERO QUE ME LA JUSTIFIQUES. NOTA: SI HAY IA, TENDRAS UN PROBLEMA, You know what I mean.!

In [1]:
import pandas as pd
import numpy as np

# --- CÓDIGO DE GENERACIÓN DE DATOS (NO MODIFICAR) ---

data = {
    'sensor_id': ['S-01', 'S-02', 'S-01', 'S-03', 'S-04', 'S-01', 'S-05', 'S-02'],
    'timestamp': [
        '2024-02-10 09:00', '2024-02-10 09:00', 
        '2024-02-10 09:15', # S-01 lectura correcta
        '2024-02-10 09:15', 
        '2024-02-10 09:15', 
        '2024-02-10 09:00', # S-01 DUPLICADO (mismo timestamp que el primero)
        None,               # S-05 Timestamp perdido
        '2024-02-10 09:30'
    ],
    'temperatura': [22.5, 21.0, None, 23.5, 20.0, 22.5, 19.5, 999.0], # 999.0 es un outlier brutal
    'humedad': [45, 50, 44, 48, None, 45, 52, -100] # -100 humedad imposible
}

df_iot = pd.DataFrame(data)
print("--- Dataset Raw (Sucio) ---")
display(df_iot)

--- Dataset Raw (Sucio) ---


,sensor_id,timestamp,temperatura,humedad
0,S-01,2024-02-10 09:00,22.5,45.0
1,S-02,2024-02-10 09:00,21.0,50.0
2,S-01,2024-02-10 09:15,NaN,44.0
3,S-03,2024-02-10 09:15,23.5,48.0
4,S-04,2024-02-10 09:15,20.0,NaN
5,S-01,2024-02-10 09:00,22.5,45.0
6,S-05,None,19.5,52.0
7,S-02,2024-02-10 09:30,999.0,-100.0


---

### Ejercicio 1: Completitud
**Problema:** Hay sensores que han fallado y no han enviado temperatura o humedad (`None`/`NaN`). También hay un timestamp perdido.

**Tarea:** 
1.  Identifica cuántos valores nulos hay en cada columna.
2.  Elimina las filas que no tengan `timestamp` (es crítico).
3.  Para `temperatura` y `humedad`, decide si borrar o rellenar (puedes usar la media o un valor fijo).

In [ ]:
# Primero miro cuantos nulos hay en cada columna para ver el panorama
print("Nulos por columna:")
print(df_iot.isnull().sum())
print()

# El timestamp es CRITICO: sin el no se cuando se tomo la medicion, no puedo
# ordenar cronologicamente ni cruzar con otros sensores. No tiene sentido
# inventarse una fecha, asi que elimino esas filas directamente.
# (En clase vimos que si un campo es clave para la trazabilidad, se descarta)
df_iot = df_iot.dropna(subset=["timestamp"])

# Para temperatura y humedad es distinto: son magnitudes fisicas continuas.
# En un edificio las condiciones no cambian bruscamente en 15 min, asi que
# tiene sentido rellenar con la media. Si borro la fila entera del S-01 a las
# 09:15, pierdo su humedad (44.0) que si era valida, y no tiene sentido.
media_temp = df_iot["temperatura"].mean()
media_hum = df_iot["humedad"].mean()
print(f"Media temp: {media_temp:.2f} | Media humedad: {media_hum:.2f}")

df_iot["temperatura"] = df_iot["temperatura"].fillna(media_temp)
df_iot["humedad"] = df_iot["humedad"].fillna(media_hum)

print("\nDataset tras tratar completitud:")
display(df_iot)

---

### Ejercicio 2: Consistencia
**Problema:** La red a veces reenvía paquetes. El sensor `S-01` parece tener una lectura duplicada a las `09:00`.

**Tarea:** 
1.  Detecta los duplicados basándote en `sensor_id` y `timestamp`.
2.  Elimínalos manteniendo solo la primera ocurrencia.

In [ ]:
# En redes IoT pasa mucho que la red reenvia paquetes cuando no recibe
# confirmacion (ACK). El S-01 a las 09:00 aparece duplicado: misma lectura
# dos veces porque la red no confirmo y el sensor lo mando otra vez.
#
# Uso sensor_id + timestamp como clave porque un mismo sensor no puede
# medir dos cosas distintas en el mismo instante. Si coinciden ambos
# campos, es un reenvio seguro.
duplicados = df_iot.duplicated(subset=["sensor_id", "timestamp"], keep="first")
print(f"Duplicados encontrados: {duplicados.sum()}")
display(df_iot[duplicados])

# Me quedo con la primera ocurrencia (la original)
df_iot = df_iot.drop_duplicates(subset=["sensor_id", "timestamp"], keep="first")
print("\nDataset sin duplicados:")
display(df_iot)

---

### Ejercicio 3: Precisión
**Problema:** 
*   El sensor `S-02` ha marcado **999.0°C**. ¡El edificio estaría en llamas!
*   La humedad de **-100%** es físicamente imposible.

**Tarea:** 
1.  Filtra el DataFrame para eliminar (o corregir) estos valores imposibles.
    *   Temperatura válida: entre -10 y 50 ºC
    *   Humedad válida: entre 0 y 100 %

In [ ]:
# 999 grados en una oficina? El sensor se descalibro o mando basura.
# -100% de humedad no existe fisicamente (va de 0 a 100).
# Esto es un tema de precision: el dato tiene formato correcto (es float)
# pero no representa la realidad.
#
# Los elimino en vez de corregirlos porque no tengo forma de saber cual
# era el valor real. Si pongo la media estaria inventandome datos, y es
# peor tener un dato falso que no tener dato. Ademas si dejo el 999, la
# media de temperatura se dispara y el modelo de consumo daria predicciones absurdas.
print("Filas con valores imposibles:")
fuera = df_iot[(df_iot["temperatura"] < -10) | (df_iot["temperatura"] > 50) | 
               (df_iot["humedad"] < 0) | (df_iot["humedad"] > 100)]
display(fuera)

# Aplico los rangos del enunciado: temp [-10, 50], humedad [0, 100]
df_iot = df_iot[(df_iot["temperatura"] >= -10) & (df_iot["temperatura"] <= 50) &
               (df_iot["humedad"] >= 0) & (df_iot["humedad"] <= 100)]
print("\nDataset limpio:")
display(df_iot)

---

### Resultado Final
Muestra cómo ha quedado el DataFrame limpio y cuántas filas tiene ahora.

In [ ]:
print("=" * 50)
print("DATASET LIMPIO FINAL")
print("=" * 50)
display(df_iot)
print(f"\nFilas finales: {len(df_iot)} (de 8 originales)")
print(df_iot[["temperatura", "humedad"]].describe())